# Source Data Model in full refresh

Dit notebook voert de Full Refresh strategie uit:
- **Stap 1**: Alle SDM tabellen LEEGMAKEN (DELETE)
- **Stap 2**: Data van alle 5 operationele DBs inlezen en in SDM vullen met INSERT

**Let op**: Draai dit notebook helemaal uit (van boven naar beneden) voor volledige refresh!

In [ ]:
# 1. IMPORTS EN INSTELLINGEN
import sqlite3
import pandas as pd
from pathlib import Path

# Dit is de map waar alle DBs staan
DB_MAP = Path(".")  # Huidige map (Week 3 folder)

# Database paden
DB_ACCESSOIREVERKOOP = DB_MAP / "BikeToDrive_1_Accessoireverkoop.db"
DB_FIETSVERKOOP = DB_MAP / "BikeToDrive_2_Fietsverkoop.db"
DB_ONDERHOUD = DB_MAP / "BikeToDrive_3_Onderhoud.db"
DB_ACCESSOIRE_INKOOP = DB_MAP / "BikeToDrive_4_Accessoire_Inkoop.db"
DB_FIETS_INKOOP = DB_MAP / "BikeToDrive_5_Fiets_Inkoop.db"
DB_SDM = DB_MAP / "BikeToDrive_6 _SourceDataModel.db"

print("✓ Imports gedaan")
print(f"✓ SDM bestand: {DB_SDM}")


In [ ]:
# 2. RESET FUNCTIE - Alle SDM tabellen leegmaken (DELETE)

def reset_sdm_tables(cursor):
    """
    Verwijdert alle data uit SDM tabellen.
    Dit is de 'reset-knop' voor volledige refresh.
    """
    
    # Alle tabellen in het SDM (in volgorde van toevoegingen)
    alle_tabellen = [
        # Accessoireverkoop
        "Accessoire_Verkoop_Leverancier",
        "Accessoire_Verkoop_Accessoire",
        "Accessoire_Verkoop_Filiaal",
        "Accessoire_Verkoop_Monteur",
        "Accessoire_Verkoop_Klant",
        "Accessoire_Verkoop",
        
        # Fietsverkoop
        "Fiets_Verkoop_Fabrikant",
        "Fiets_Verkoop_Fiets",
        "Fiets_Verkoop_Filiaal",
        "Fiets_Verkoop_Monteur",
        "Fiets_Verkoop_Klant",
        "Fiets_Verkoop",
        
        # Onderhoud
        "Onderhoud_Fabrikant",
        "Onderhoud_Fiets",
        "Onderhoud_Filiaal",
        "Onderhoud_Monteur",
        "Onderhoud",
        
        # Accessoire Inkoop
        "Accessoire_Inkoop_Leverancier",
        "Accessoire_Inkoop_Accessoire",
        "Accessoire_Inkoop",
        
        # Fiets Inkoop
        "Fiets_Inkoop_Fabrikant",
        "Fiets_Inkoop_Fiets",
        "Fiets_Inkoop"
    ]
    
    # Draai de lijst om - de laatste toegevoegde tabel moet als eerste worden leeggmaakt!
    # Dit is belangrijk vanwege FOREIGN KEYS (geen referenties mogen nog bestaan)
    alle_tabellen = alle_tabellen[::-1]
    
    for tabel in alle_tabellen:
        delete_statement = f"DELETE FROM {tabel};"
        
        try:
            cursor.execute(delete_statement)
            print(f"✓ {tabel}: data verwijderd")
        except sqlite3.Error as e:
            print(f"✗ {tabel}: FOUT - {e}")

print("✓ Reset functie gedefinieerd")

In [ ]:
# 3. LOAD & INSERT FUNCTIE - Database 1: Accessoireverkoop

def load_en_insert_accessoireverkoop(cursor, conn):
    """Voert data van Accessoireverkoop DB in het SDM in (Full Refresh)"""
    
    print("\n--- START: Accessoireverkoop laden ---")
    
    # Connectie met operationele DB
    conn_bron = sqlite3.connect(DB_ACCESSOIREVERKOOP)
    
    # ===== LEVERANCIERS =====
    print("Loading: Leveranciers...")
    df_leveranciers = pd.read_sql_query("SELECT * FROM Leverancier", conn_bron)
    
    for row in df_leveranciers.iterrows():
        insert_statement = f"INSERT INTO Accessoire_Verkoop_Leverancier VALUES ("
        insert_statement += str(row[1]['leveranciernr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['adres']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['woonplaats']).replace("'", "''") + "');"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij leverancier: {insert_statement}")
    
    print(f"✓ {len(df_leveranciers)} leveranciers ingevoerd")
    conn.commit()
    
    # ===== ACCESSOIRES =====
    print("Loading: Accessoires...")
    df_accessoires = pd.read_sql_query("SELECT * FROM Accessoire", conn_bron)
    
    for row in df_accessoires.iterrows():
        insert_statement = f"INSERT INTO Accessoire_Verkoop_Accessoire VALUES ("
        insert_statement += str(row[1]['accessoirenr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += str(row[1]['standaardprijs']) + ", "
        insert_statement += str(row[1]['inkoopprijs']) + ", "
        insert_statement += "'" + str(row[1]['soort']).replace("'", "''") + "', "
        insert_statement += str(row[1]['leverancier']) + ");"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij accessoire: {insert_statement}")
    
    print(f"✓ {len(df_accessoires)} accessoires ingevoerd")
    conn.commit()
    
    # ===== FILIALEN =====
    print("Loading: Filialen...")
    df_filialen = pd.read_sql_query("SELECT * FROM Filiaal", conn_bron)
    
    for row in df_filialen.iterrows():
        insert_statement = f"INSERT INTO Accessoire_Verkoop_Filiaal VALUES ("
        insert_statement += str(row[1]['filiaalnr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['adres']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['provincie']).replace("'", "''") + "');"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij filiaal: {insert_statement}")
    
    print(f"✓ {len(df_filialen)} filialen ingevoerd")
    conn.commit()
    
    # ===== MONTEURS =====
    print("Loading: Monteurs...")
    df_monteurs = pd.read_sql_query("SELECT * FROM Monteur", conn_bron)
    
    for row in df_monteurs.iterrows():
        insert_statement = f"INSERT INTO Accessoire_Verkoop_Monteur VALUES ("
        insert_statement += str(row[1]['monteurnr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['woonplaats']).replace("'", "''") + "', "
        insert_statement += str(row[1]['uurloon']) + ", "
        insert_statement += str(row[1]['filiaal']) + ");"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij monteur: {insert_statement}")
    
    print(f"✓ {len(df_monteurs)} monteurs ingevoerd")
    conn.commit()
    
    # ===== KLANTEN =====
    print("Loading: Klanten...")
    df_klanten = pd.read_sql_query("SELECT * FROM Klant", conn_bron)
    
    for row in df_klanten.iterrows():
        insert_statement = f"INSERT INTO Accessoire_Verkoop_Klant VALUES ("
        insert_statement += str(row[1]['klantnr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['adres']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['woonplaats']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['geslacht']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['geboortedatum']) + "');"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij klant: {insert_statement}")
    
    print(f"✓ {len(df_klanten)} klanten ingevoerd")
    conn.commit()
    
    # ===== VERKOPEN =====
    print("Loading: Verkopen...")
    df_verkopen = pd.read_sql_query("SELECT * FROM Accessoire_Verkoop", conn_bron)
    
    for row in df_verkopen.iterrows():
        insert_statement = f"INSERT INTO Accessoire_Verkoop VALUES ("
        insert_statement += str(row[1]['accessoire_verkoopnr']) + ", "
        insert_statement += "'" + str(row[1]['datum']) + "', "
        insert_statement += str(row[1]['aantal']) + ", "
        insert_statement += str(row[1]['verkoopprijs']) + ", "
        insert_statement += str(row[1]['klant']) + ", "
        insert_statement += str(row[1]['accessoire']) + ", "
        insert_statement += str(row[1]['monteur']) + ");"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij verkoop: {insert_statement}")
    
    print(f"✓ {len(df_verkopen)} verkopen ingevoerd")
    conn.commit()
    
    conn_bron.close()
    print("✓ Accessoireverkoop klaar!\n")

print("✓ Load & Insert functie Accessoireverkoop gedefinieerd")

In [ ]:
# 4. LOAD & INSERT FUNCTIE - Database 2: Fietsverkoop

def load_en_insert_fietsverkoop(cursor, conn):
    """Voert data van Fietsverkoop DB in het SDM in (Full Refresh)"""
    
    print("\n--- START: Fietsverkoop laden ---")
    
    conn_bron = sqlite3.connect(DB_FIETSVERKOOP)
    
    # ===== FABRIKANTEN =====
    print("Loading: Fabrikanten...")
    df_fabrikanten = pd.read_sql_query("SELECT * FROM Fabrikant", conn_bron)
    
    for row in df_fabrikanten.iterrows():
        insert_statement = f"INSERT INTO Fiets_Verkoop_Fabrikant VALUES ("
        insert_statement += str(row[1]['fabrikantnr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['adres']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['plaats']).replace("'", "''") + "');"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij fabrikant: {insert_statement}")
    
    print(f"✓ {len(df_fabrikanten)} fabrikanten ingevoerd")
    conn.commit()
    
    # ===== FIETSEN =====
    print("Loading: Fietsen...")
    df_fietsen = pd.read_sql_query("SELECT * FROM Fiets", conn_bron)
    
    for row in df_fietsen.iterrows():
        insert_statement = f"INSERT INTO Fiets_Verkoop_Fiets VALUES ("
        insert_statement += str(row[1]['fietsnr']) + ", "
        insert_statement += "'" + str(row[1]['soort']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['merk']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['type']).replace("'", "''") + "', "
        insert_statement += str(row[1]['standaardprijs']) + ", "
        insert_statement += str(row[1]['inkoopprijs']) + ", "
        insert_statement += "'" + str(row[1]['kleur']).replace("'", "''") + "', "
        insert_statement += str(row[1]['fabrikant']) + ");"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij fiets: {insert_statement}")
    
    print(f"✓ {len(df_fietsen)} fietsen ingevoerd")
    conn.commit()
    
    # ===== FILIALEN =====
    print("Loading: Filialen...")
    df_filialen = pd.read_sql_query("SELECT * FROM Filiaal", conn_bron)
    
    for row in df_filialen.iterrows():
        insert_statement = f"INSERT INTO Fiets_Verkoop_Filiaal VALUES ("
        insert_statement += str(row[1]['filiaalnr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['adres']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['provincie']).replace("'", "''") + "');"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij filiaal: {insert_statement}")
    
    print(f"✓ {len(df_filialen)} filialen ingevoerd")
    conn.commit()
    
    # ===== MONTEURS =====
    print("Loading: Monteurs...")
    df_monteurs = pd.read_sql_query("SELECT * FROM Monteur", conn_bron)
    
    for row in df_monteurs.iterrows():
        insert_statement = f"INSERT INTO Fiets_Verkoop_Monteur VALUES ("
        insert_statement += str(row[1]['monteurnr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['woonplaats']).replace("'", "''") + "', "
        insert_statement += str(row[1]['uurloon']) + ", "
        insert_statement += str(row[1]['filiaal']) + ");"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij monteur: {insert_statement}")
    
    print(f"✓ {len(df_monteurs)} monteurs ingevoerd")
    conn.commit()
    
    # ===== KLANTEN =====
    print("Loading: Klanten...")
    df_klanten = pd.read_sql_query("SELECT * FROM Klant", conn_bron)
    
    for row in df_klanten.iterrows():
        insert_statement = f"INSERT INTO Fiets_Verkoop_Klant VALUES ("
        insert_statement += str(row[1]['klantnr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['adres']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['woonplaats']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['geslacht']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['geboortedatum']) + "');"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij klant: {insert_statement}")
    
    print(f"✓ {len(df_klanten)} klanten ingevoerd")
    conn.commit()
    
    # ===== VERKOPEN =====
    print("Loading: Verkopen...")
    df_verkopen = pd.read_sql_query("SELECT * FROM Fiets_Verkoop", conn_bron)
    
    for row in df_verkopen.iterrows():
        insert_statement = f"INSERT INTO Fiets_Verkoop VALUES ("
        insert_statement += str(row[1]['fiets_verkoopnr']) + ", "
        insert_statement += "'" + str(row[1]['datum']) + "', "
        insert_statement += str(row[1]['aantal']) + ", "
        insert_statement += str(row[1]['verkoopprijs']) + ", "
        insert_statement += str(row[1]['klant']) + ", "
        insert_statement += str(row[1]['fiets']) + ", "
        insert_statement += str(row[1]['monteur']) + ");"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij verkoop: {insert_statement}")
    
    print(f"✓ {len(df_verkopen)} verkopen ingevoerd")
    conn.commit()
    
    conn_bron.close()
    print("✓ Fietsverkoop klaar!\n")

print("✓ Load & Insert functie Fietsverkoop gedefinieerd")

In [ ]:
# 5. LOAD & INSERT FUNCTIE - Database 3: Onderhoud

def load_en_insert_onderhoud(cursor, conn):
    """Voert data van Onderhoud DB in het SDM in (Full Refresh)"""
    
    print("\n--- START: Onderhoud laden ---")
    
    conn_bron = sqlite3.connect(DB_ONDERHOUD)
    
    # ===== FABRIKANTEN =====
    print("Loading: Fabrikanten...")
    df_fabrikanten = pd.read_sql_query("SELECT * FROM Fabrikant", conn_bron)
    
    for row in df_fabrikanten.iterrows():
        insert_statement = f"INSERT INTO Onderhoud_Fabrikant VALUES ("
        insert_statement += str(row[1]['fabrikantnr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['adres']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['plaats']).replace("'", "''") + "');"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij fabrikant: {insert_statement}")
    
    print(f"✓ {len(df_fabrikanten)} fabrikanten ingevoerd")
    conn.commit()
    
    # ===== FIETSEN =====
    print("Loading: Fietsen...")
    df_fietsen = pd.read_sql_query("SELECT * FROM Fiets", conn_bron)
    
    for row in df_fietsen.iterrows():
        insert_statement = f"INSERT INTO Onderhoud_Fiets VALUES ("
        insert_statement += str(row[1]['fietsnr']) + ", "
        insert_statement += "'" + str(row[1]['soort']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['merk']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['type']).replace("'", "''") + "', "
        insert_statement += str(row[1]['standaardprijs']) + ", "
        insert_statement += str(row[1]['inkoopprijs']) + ", "
        insert_statement += "'" + str(row[1]['kleur']).replace("'", "''") + "', "
        insert_statement += str(row[1]['fabrikant']) + ");"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij fiets: {insert_statement}")
    
    print(f"✓ {len(df_fietsen)} fietsen ingevoerd")
    conn.commit()
    
    # ===== FILIALEN =====
    print("Loading: Filialen...")
    df_filialen = pd.read_sql_query("SELECT * FROM Filiaal", conn_bron)
    
    for row in df_filialen.iterrows():
        insert_statement = f"INSERT INTO Onderhoud_Filiaal VALUES ("
        insert_statement += str(row[1]['filiaalnr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['adres']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['provincie']).replace("'", "''") + "');"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij filiaal: {insert_statement}")
    
    print(f"✓ {len(df_filialen)} filialen ingevoerd")
    conn.commit()
    
    # ===== MONTEURS =====
    print("Loading: Monteurs...")
    df_monteurs = pd.read_sql_query("SELECT * FROM Monteur", conn_bron)
    
    for row in df_monteurs.iterrows():
        insert_statement = f"INSERT INTO Onderhoud_Monteur VALUES ("
        insert_statement += str(row[1]['monteurnr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['woonplaats']).replace("'", "''") + "', "
        insert_statement += str(row[1]['uurloon']) + ", "
        insert_statement += str(row[1]['filiaal']) + ");"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij monteur: {insert_statement}")
    
    print(f"✓ {len(df_monteurs)} monteurs ingevoerd")
    conn.commit()
    
    # ===== ONDERHOUDSWERKZAAMHEDEN =====
    print("Loading: Onderhoudsrekenen...")
    df_onderhoud = pd.read_sql_query("SELECT * FROM Onderhoud", conn_bron)
    
    for row in df_onderhoud.iterrows():
        insert_statement = f"INSERT INTO Onderhoud VALUES ("
        insert_statement += str(row[1]['onderhoudnr']) + ", "
        insert_statement += "'" + str(row[1]['datum']) + "', "
        insert_statement += "'" + str(row[1]['starttijd']) + "', "
        insert_statement += "'" + str(row[1]['eindtijd']) + "', "
        insert_statement += str(row[1]['fiets']) + ", "
        insert_statement += str(row[1]['monteur']) + ");"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij onderhoud: {insert_statement}")
    
    print(f"✓ {len(df_onderhoud)} onderhoudsrekenen ingevoerd")
    conn.commit()
    
    conn_bron.close()
    print("✓ Onderhoud klaar!\n")

print("✓ Load & Insert functie Onderhoud gedefinieerd")

In [ ]:
# 6. LOAD & INSERT FUNCTIE - Database 4: Accessoire Inkoop

def load_en_insert_accessoire_inkoop(cursor, conn):
    """Voert data van Accessoire Inkoop DB in het SDM in (Full Refresh)"""
    
    print("\n--- START: Accessoire Inkoop laden ---")
    
    conn_bron = sqlite3.connect(DB_ACCESSOIRE_INKOOP)
    
    # ===== LEVERANCIERS =====
    print("Loading: Leveranciers...")
    df_leveranciers = pd.read_sql_query("SELECT * FROM Leverancier", conn_bron)
    
    for row in df_leveranciers.iterrows():
        insert_statement = f"INSERT INTO Accessoire_Inkoop_Leverancier VALUES ("
        insert_statement += str(row[1]['leveranciernr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['adres']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['woonplaats']).replace("'", "''") + "');"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij leverancier: {insert_statement}")
    
    print(f"✓ {len(df_leveranciers)} leveranciers ingevoerd")
    conn.commit()
    
    # ===== ACCESSOIRES =====
    print("Loading: Accessoires...")
    df_accessoires = pd.read_sql_query("SELECT * FROM Accessoire", conn_bron)
    
    for row in df_accessoires.iterrows():
        insert_statement = f"INSERT INTO Accessoire_Inkoop_Accessoire VALUES ("
        insert_statement += str(row[1]['accessoirenr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += str(row[1]['standaardprijs']) + ", "
        insert_statement += str(row[1]['inkoopprijs']) + ", "
        insert_statement += "'" + str(row[1]['soort']).replace("'", "''") + "', "
        insert_statement += str(row[1]['leverancier']) + ");"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij accessoire: {insert_statement}")
    
    print(f"✓ {len(df_accessoires)} accessoires ingevoerd")
    conn.commit()
    
    # ===== INKOOP =====
    print("Loading: Inkoop...")
    df_inkoop = pd.read_sql_query("SELECT * FROM Accessoire_Inkoop", conn_bron)
    
    for row in df_inkoop.iterrows():
        insert_statement = f"INSERT INTO Accessoire_Inkoop VALUES ("
        insert_statement += str(row[1]['inkoopnr']) + ", "
        insert_statement += str(row[1]['inkoopmaand']) + ", "
        insert_statement += str(row[1]['inkoopjaar']) + ", "
        insert_statement += str(row[1]['aantal']) + ", "
        insert_statement += str(row[1]['accessoire']) + ");"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij inkoop: {insert_statement}")
    
    print(f"✓ {len(df_inkoop)} inkoop ingevoerd")
    conn.commit()
    
    conn_bron.close()
    print("✓ Accessoire Inkoop klaar!\n")

print("✓ Load & Insert functie Accessoire Inkoop gedefinieerd")

In [ ]:
# 7. LOAD & INSERT FUNCTIE - Database 5: Fiets Inkoop

def load_en_insert_fiets_inkoop(cursor, conn):
    """Voert data van Fiets Inkoop DB in het SDM in (Full Refresh)"""
    
    print("\n--- START: Fiets Inkoop laden ---")
    
    conn_bron = sqlite3.connect(DB_FIETS_INKOOP)
    
    # ===== FABRIKANTEN =====
    print("Loading: Fabrikanten...")
    df_fabrikanten = pd.read_sql_query("SELECT * FROM Fabrikant", conn_bron)
    
    for row in df_fabrikanten.iterrows():
        insert_statement = f"INSERT INTO Fiets_Inkoop_Fabrikant VALUES ("
        insert_statement += str(row[1]['fabrikantnr']) + ", "
        insert_statement += "'" + str(row[1]['naam']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['adres']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['plaats']).replace("'", "''") + "');"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij fabrikant: {insert_statement}")
    
    print(f"✓ {len(df_fabrikanten)} fabrikanten ingevoerd")
    conn.commit()
    
    # ===== FIETSEN =====
    print("Loading: Fietsen...")
    df_fietsen = pd.read_sql_query("SELECT * FROM Fiets", conn_bron)
    
    for row in df_fietsen.iterrows():
        insert_statement = f"INSERT INTO Fiets_Inkoop_Fiets VALUES ("
        insert_statement += str(row[1]['fietsnr']) + ", "
        insert_statement += "'" + str(row[1]['soort']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['merk']).replace("'", "''") + "', "
        insert_statement += "'" + str(row[1]['type']).replace("'", "''") + "', "
        insert_statement += str(row[1]['standaardprijs']) + ", "
        insert_statement += str(row[1]['inkoopprijs']) + ", "
        insert_statement += "'" + str(row[1]['kleur']).replace("'", "''") + "', "
        insert_statement += str(row[1]['fabrikant']) + ");"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij fiets: {insert_statement}")
    
    print(f"✓ {len(df_fietsen)} fietsen ingevoerd")
    conn.commit()
    
    # ===== INKOOP =====
    print("Loading: Inkoop...")
    df_inkoop = pd.read_sql_query("SELECT * FROM Fiets_Inkoop", conn_bron)
    
    for row in df_inkoop.iterrows():
        insert_statement = f"INSERT INTO Fiets_Inkoop VALUES ("
        insert_statement += str(row[1]['inkoopnr']) + ", "
        insert_statement += str(row[1]['inkoopmaand']) + ", "
        insert_statement += str(row[1]['inkoopjaar']) + ", "
        insert_statement += str(row[1]['aantal']) + ", "
        insert_statement += str(row[1]['fiets']) + ");"
        
        try:
            cursor.execute(insert_statement)
        except sqlite3.Error:
            print(f"FOUT bij inkoop: {insert_statement}")
    
    print(f"✓ {len(df_inkoop)} inkoop ingevoerd")
    conn.commit()
    
    conn_bron.close()
    print("✓ Fiets Inkoop klaar!\n")

print("✓ Load & Insert functie Fiets Inkoop gedefinieerd")

In [ ]:
# 8. MAIN SCRIPT - Full Refresh uitvoeren

print("=" * 60)
print("STARTEN MET FULL REFRESH VAN SOURCE DATA MODEL")
print("=" * 60)

aenc_sdm_conn = None  # Initialize eerst op None voor veiligheid

try:
    # Verbinding met SDM database
    aenc_sdm_conn = sqlite3.connect(DB_SDM)
    aenc_sdm_cursor = aenc_sdm_conn.cursor()
    
    # STAP 1: RESET - Alle tabellen leegmaken
    print("\n🔄 STAP 1: Reset (DELETE all tables)")
    print("-" * 60)
    reset_sdm_tables(aenc_sdm_cursor)
    aenc_sdm_conn.commit()
    print("✓ Alle tabellen geleegd en gecommit!")
    
    # STAP 2: DATA VULLEN - Van alle operationele databases
    print("\n📥 STAP 2: Data laden van operationele databases")
    print("-" * 60)
    
    load_en_insert_accessoireverkoop(aenc_sdm_cursor, aenc_sdm_conn)
    load_en_insert_fietsverkoop(aenc_sdm_cursor, aenc_sdm_conn)
    load_en_insert_onderhoud(aenc_sdm_cursor, aenc_sdm_conn)
    load_en_insert_accessoire_inkoop(aenc_sdm_cursor, aenc_sdm_conn)
    load_en_insert_fiets_inkoop(aenc_sdm_cursor, aenc_sdm_conn)
    
    # Klaar!
    print("=" * 60)
    print("✓✓✓ FULL REFRESH SUCCESVOL AFGEROND ✓✓✓")
    print("=" * 60)
    print(f"✓ SDM database: {DB_SDM}")
    print(f"✓ Alle data is ingevoerd en gecommit!")
    
    if aenc_sdm_conn:
        aenc_sdm_conn.close()
    
except Exception as e:
    print(f"\n✗✗✗ FOUT OPGETREDEN ✗✗✗")
    print(f"Foutmelding: {e}")
    if aenc_sdm_conn:
        aenc_sdm_conn.rollback()
        aenc_sdm_conn.close()




##  Reset SDM functie uitvoeren


In [ ]:
# Reset SDM

print("=" * 60)
print("RESET ONLY - SDM LEEGMAKEN (geen data inladen)")
print("=" * 60)

aenc_sdm_conn = None

try:
    aenc_sdm_conn = sqlite3.connect(DB_SDM)
    aenc_sdm_cursor = aenc_sdm_conn.cursor()
    
    print("\n🔄 Reset uitvoeren...")
    print("-" * 60)
    reset_sdm_tables(aenc_sdm_cursor)
    aenc_sdm_conn.commit()
    
    print("-" * 60)
    print("✓✓✓ RESET VOLTOOID - SDM IS LEEG ✓✓✓")
    print("=" * 60)
    
    if aenc_sdm_conn:
        aenc_sdm_conn.close()
    
except Exception as e:
    print(f"\n✗✗✗ FOUT OPGETREDEN ✗✗✗")
    print(f"Foutmelding: {e}")
    if aenc_sdm_conn:
        aenc_sdm_conn.rollback()
        aenc_sdm_conn.close()



##  Check of data correct in SDM zit


In [13]:
# 9. VERIFICATIE - Check of data correct ingevoerd is

print("\n" + "=" * 60)
print("VERIFICATIE: Controle aantal rijen per tabel")
print("=" * 60)

conn_sdm = sqlite3.connect(DB_SDM)

# Query per tabel (een paar belangrijke tabellen)
test_queries = [
    ("Accessoire_Verkoop", "SELECT COUNT(*) as aantal FROM Accessoire_Verkoop"),
    ("Fiets_Verkoop", "SELECT COUNT(*) as aantal FROM Fiets_Verkoop"),
    ("Onderhoud", "SELECT COUNT(*) as aantal FROM Onderhoud"),
    ("Accessoire_Inkoop", "SELECT COUNT(*) as aantal FROM Accessoire_Inkoop"),
    ("Fiets_Inkoop", "SELECT COUNT(*) as aantal FROM Fiets_Inkoop"),
    ("Accessoire_Verkoop_Klant", "SELECT COUNT(*) as aantal FROM Accessoire_Verkoop_Klant"),
    ("Fiets_Verkoop_Klant", "SELECT COUNT(*) as aantal FROM Fiets_Verkoop_Klant"),
]

print("\nAantal rijen per tabel:")
print("-" * 60)

for tabel_naam, query in test_queries:
    try:
        result = pd.read_sql_query(query, conn_sdm)
        aantal = result['aantal'][0]
        print(f"  {tabel_naam}: {aantal} rijen")
    except Exception as e:
        print(f"  {tabel_naam}: FOUT - {e}")

print("-" * 60)
print("✓ Verificatie afgerond!")

conn_sdm.close()


VERIFICATIE: Controle aantal rijen per tabel

Aantal rijen per tabel:
------------------------------------------------------------
  Accessoire_Verkoop: 0 rijen
  Fiets_Verkoop: 0 rijen
  Onderhoud: 0 rijen
  Accessoire_Inkoop: 0 rijen
  Fiets_Inkoop: 0 rijen
  Accessoire_Verkoop_Klant: 0 rijen
  Fiets_Verkoop_Klant: 0 rijen
------------------------------------------------------------
✓ Verificatie afgerond!


In [ ]:
# 10. TEST - Voeg een rij toe aan Fiets_Inkoop en check

print("\n" + "=" * 60)
print("TEST: Voeg rij toe aan Fiets_Inkoop")
print("=" * 60)

try:
    conn = sqlite3.connect(DB_FIETS_INKOOP)
    cursor = conn.cursor()
    
    # Tel rijen VOOR
    cursor.execute("SELECT COUNT(*) FROM Fiets_Inkoop")
    count_before = cursor.fetchone()[0]
    print(f"\nRijen VÓÓR toevoeging: {count_before}")
    
    # Haal alle kolommen van de tabel op
    cursor.execute("PRAGMA table_info(Fiets_Inkoop)")
    columns = cursor.fetchall()
    print(f"Kolommen in Fiets_Inkoop: {[col[1] for col in columns]}")
    
    # Haal hoogste inkoopnr op
    cursor.execute("SELECT COALESCE(MAX(inkoopnr), 0) FROM Fiets_Inkoop")
    max_nr = cursor.fetchone()[0]
    new_nr = max_nr + 1
    print(f"Highest inkoopnr: {max_nr} → New: {new_nr}")
    
    # Voeg rij toe
    insert_sql = f"INSERT INTO Fiets_Inkoop (inkoopnr, inkoopmaand, inkoopjaar, aantal, fiets) VALUES ({new_nr}, 3, 2024, 5, 1)"
    print(f"\nExecuting: {insert_sql}")
    
    cursor.execute(insert_sql)
    conn.commit()
    print(f"✓ Rij succesvol toegevoegd")
    
    # Tel rijen NA
    cursor.execute("SELECT COUNT(*) FROM Fiets_Inkoop")
    count_after = cursor.fetchone()[0]
    print(f"Rijen NA toevoeging: {count_after}")
    print(f"Verschil: +{count_after - count_before} rij")
    
    conn.close()
    print("=" * 60)
    
except Exception as e:
    print(f"\n✗ FOUT OPGETREDEN: {e}")
    print(f"Type fout: {type(e).__name__}")
    import traceback
    traceback.print_exc()
    try:
        conn.rollback()
        conn.close()
    except:
        pass



TEST: Voeg rij toe aan Fiets_Inkoop

Rijen VÓÓR toevoeging: 100
✗ Fout bij invoegen: database is locked
Rijen NA toevoeging: 100
Verschil: +0 rij
